In [3]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, recall_score, precision_score, average_precision_score
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

In [4]:
df = pd.read_csv('../data/creditcard.csv')

X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

Training set: (227845, 30)
Test set: (56962, 30)


In [5]:
mlflow.set_experiment("fraud-detection")
print("MLflow experiment set: fraud-detection")

2026/07/23 18:48:21 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/23 18:48:21 INFO mlflow.store.db.utils: Updating database tables
2026/07/23 18:48:21 INFO mlflow.tracking.fluent: Experiment with name 'fraud-detection' does not exist. Creating a new experiment.


MLflow experiment set: fraud-detection


In [6]:
models = {
    'LogisticRegression': {
        'model': LogisticRegression(max_iter=1000, random_state=42),
        'params': {
            'max_iter': 1000,
            'random_state': 42
        }
    },
    'RandomForest': {
        'model': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        'params': {
            'n_estimators': 100,
            'random_state': 42
        }
    },
    'XGBoost': {
        'model': XGBClassifier(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.1,
            subsample=0.8,
            random_state=42,
            eval_metric='logloss'
        ),
        'params': {
            'n_estimators': 200,
            'max_depth': 4,
            'learning_rate': 0.1,
            'subsample': 0.8
        }
    }
}

In [12]:
for name, config in models.items():
    print(f"\nTraining and logging {name}...")
    
    with mlflow.start_run(run_name=name):
        # build pipeline
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('smote', SMOTE(random_state=42)),
            ('model', config['model'])
        ])
        
        # train
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        y_scores = pipeline.predict_proba(X_test)[:, 1]
        
        # calculate metrics
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        auc_pr = average_precision_score(y_test, y_scores)
        
        # log parameters to MLflow
        mlflow.log_params(config['params'])
        
        # log metrics to MLflow
        mlflow.log_metrics({
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'auc_pr': auc_pr
        })
        
        # log the model itself
        mlflow.sklearn.log_model(
        pipeline,
        name,
        skops_trusted_types=[
            'imblearn.over_sampling._smote.base.SMOTE',
            'imblearn.pipeline.Pipeline',
            'xgboost.core.Booster',
            'xgboost.sklearn.XGBClassifier',
            'sklearn.linear_model._logistic.LogisticRegression',
            'sklearn.ensemble._forest.RandomForestClassifier',
            'sklearn.preprocessing._data.StandardScaler'
        ]
)
        
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall:    {recall:.4f}")
        print(f"  F1:        {f1:.4f}")
        print(f"  AUC-PR:    {auc_pr:.4f}")
        print(f"  Logged to MLflow.")

print("\nAll runs logged.")


Training and logging LogisticRegression...


2026/07/23 18:59:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  Precision: 0.0578
  Recall:    0.9184
  F1:        0.1088
  AUC-PR:    0.7245
  Logged to MLflow.

Training and logging RandomForest...


2026/07/23 19:00:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  Precision: 0.8710
  Recall:    0.8265
  F1:        0.8482
  AUC-PR:    0.8723
  Logged to MLflow.

Training and logging XGBoost...


2026/07/23 19:00:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  Precision: 0.2972
  Recall:    0.8673
  F1:        0.4427
  AUC-PR:    0.8419
  Logged to MLflow.

All runs logged.


In [13]:
print("To view the MLflow UI, run this in your terminal:")
print("mlflow ui")
print("\nThen open: http://127.0.0.1:5000")
print("\nYou'll see all three runs with their parameters and metrics.")

To view the MLflow UI, run this in your terminal:
mlflow ui

Then open: http://127.0.0.1:5000

You'll see all three runs with their parameters and metrics.


In [14]:
# search for all runs in the experiment
runs = mlflow.search_runs(experiment_names=["fraud-detection"])
print(f"Total runs logged: {len(runs)}")
print("\nRuns summary:")
cols = ['tags.mlflow.runName', 'metrics.precision', 'metrics.recall', 
        'metrics.f1_score', 'metrics.auc_pr']
print(runs[cols].to_string(index=False))

Total runs logged: 14

Runs summary:
tags.mlflow.runName  metrics.precision  metrics.recall  metrics.f1_score  metrics.auc_pr
            XGBoost           0.297203        0.867347          0.442708        0.841910
       RandomForest           0.870968        0.826531          0.848168        0.872328
 LogisticRegression           0.057803        0.918367          0.108761        0.724469
            XGBoost           0.297203        0.867347          0.442708        0.841910
       RandomForest           0.870968        0.826531          0.848168        0.872328
 LogisticRegression           0.057803        0.918367          0.108761        0.724469
            XGBoost           0.297203        0.867347          0.442708        0.841910
       RandomForest           0.870968        0.826531          0.848168        0.872328
 LogisticRegression           0.057803        0.918367          0.108761        0.724469
            XGBoost           0.297203        0.867347          0.442708 